In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Cleaning data
Droping the participants with Order = "Other" as it means they did not follow the instructions appropiatley.

In [ ]:
df = pd.read_csv('data/Biathlon Metrics.tsv', sep='\t')
df.rename(columns={'Recording': 'condition'}, inplace=True)
df = df[df['Order'] != 'Other']

## Normalising the condition naming

In [ ]:
def normalize_condition(val):
    v = val.lower().strip()
    if 'fatigue1' in v or 'fatigued1' in v or 'fatigue 1' in v:
        return 'fatigue1'
    if 'fatigue2' in v or 'fatigued2' in v or 'fatigue 2' in v:
        return 'fatigue2'
    return 'no_fatigue'

df['condition'] = df['condition'].apply(normalize_condition)
print(f'{df.shape[0]} rows x {df.shape[1]} cols')
print(f'\nConditions: {df["condition"].value_counts().to_dict()}')
print(f'Orders: {df["Order"].value_counts().to_dict()}')
df.head(10)

## Flattening

Columns `.1`-`.5` = physical target positions on screen.
Each row = one interval (step in viewing sequence).
- L-R: interval i looked at physical position i
- R-L: interval i looked at physical position (6-i)

After flattening, columns `fix_1`..`fix_5` and `ttff_1`..`ttff_5` represent the 1st through 5th target viewed.

In [ ]:
def active_position(row):
    if row['Order'] == 'L-R':
        return row['Interval']
    elif row['Order'] == 'R-L':
        return 6 - row['Interval']
    return np.nan

df['active_pos'] = df.apply(active_position, axis=1)

df['fix_val'] = df.apply(lambda r: r[f'Number_of_fixations.{int(r["active_pos"])}'] if pd.notna(r['active_pos']) else np.nan, axis=1)
df['ttff_val'] = df.apply(lambda r: r[f'Time_to_first_fixation.{int(r["active_pos"])}'] if pd.notna(r['active_pos']) else np.nan, axis=1)

group_keys = ['Participant', 'condition', 'Order']
pivot_fix = df.pivot_table(index=group_keys, columns='active_pos', values='fix_val', aggfunc='first')
pivot_ttff = df.pivot_table(index=group_keys, columns='active_pos', values='ttff_val', aggfunc='first')

pivot_fix.columns = [f'fix_{int(c)}' for c in pivot_fix.columns]
pivot_ttff.columns = [f'ttff_{int(c)}' for c in pivot_ttff.columns]

df_flat = pd.concat([pivot_fix, pivot_ttff], axis=1).reset_index()

# R-L flip: pivot columns came in [5,4,3,2,1], so fix_1 = pos 5. Swap pairs.
rl = df_flat['Order'] == 'R-L'
for col in ['fix', 'ttff']:
    for i in [1, 2]:
        a, b = f'{col}_{i}', f'{col}_{6-i}'
        tmp = df_flat.loc[rl, a].copy()
        df_flat.loc[rl, a] = df_flat.loc[rl, b].values
        df_flat.loc[rl, b] = tmp.values

print(f'Flattened: {df_flat.shape[0]} rows x {df_flat.shape[1]} cols')
df_flat.head(10)

## Calculations
### Adding anticipation
How many trials have responses that are anticipatory.

In [ ]:
TTFF_THRESHOLD = 150  # ms – count how many positions have ttff below this

ttff_cols = [c for c in df_flat.columns if c.startswith('ttff_')]
df_flat['anticipation'] = (df_flat[ttff_cols] < TTFF_THRESHOLD).sum(axis=1)
df_flat[['condition', 'ttff_1', 'ttff_2', 'ttff_3', 'ttff_4', 'ttff_5', 'anticipation']].head(10)

### Adding Total fixations

In [ ]:
fix_cols = [c for c in df_flat.columns if c.startswith('fix_')]
df_flat['total_fix'] = df_flat[fix_cols].sum(axis=1)

### Adding average time to first fixation
Adding Average to first fixations with and without the first trial. On the first trial, they need to move from the fixation cross to the first target wich is a greater distance than the rest of the sacadic movments they need to do on the rest of the experiment.

In [ ]:
ttff_cols = [c for c in df_flat.columns if c.startswith('ttff_')]
df_flat['mean_ttff'] = df_flat[ttff_cols].mean(axis=1)

df_flat['mean_ttff_excl1'] = df_flat[[c for c in ttff_cols if c != 'ttff_1']].mean(axis=1)

## Final table

In [ ]:
df_flat.head(10)

## Ploting the results
Boxplot per condition of the 3 different metrics

In [ ]:
order = ['no_fatigue', 'fatigue1', 'fatigue2']
fig_box, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(axes, ['total_fix', 'mean_ttff', 'mean_ttff_excl1'],
                          ['Total fixations', 'Mean TTFF (ms)', 'Mean TTFF excl. pos1 (ms)']):
    data = [df_flat[df_flat['condition'] == c][col].dropna() for c in order]
    ax.boxplot(data, tick_labels=order, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='black'),
               medianprops=dict(color='red'))
    ax.set_title(label)
    ax.set_xticklabels(order, rotation=15)

fig_box.tight_layout()



In [ ]:
# Anticipation frequency distribution (side-by-side)
counts = {cond: df_flat[df_flat['condition'] == cond]['anticipation'].value_counts().reindex(range(6), fill_value=0) for cond in order}
x = np.arange(6)
width = 0.25
# greyscale + hatch patterns for print-friendly distinction
styles = [('white', ''), ('gray', '///'), ('black', '...')]

fig_hist, ax = plt.subplots(figsize=(8, 4))
for i, (cond, (fc, hatch)) in enumerate(zip(order, styles)):
    bars = ax.bar(x + i * width, counts[cond], width, label=cond,
                  facecolor=fc, edgecolor='black', hatch=hatch)

ax.set_xlabel('Anticipatory responses (n)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of anticipatory responses by condition')
ax.set_xticks(x + width)
ax.set_xticklabels(range(6))
ax.legend()
fig_hist.tight_layout()



# Exporting data
Exporting an excel file with one tab per each table of interest.

In [ ]:
export_cols = ['Participant', 'condition', 'Order',
               'fix_1', 'fix_2', 'fix_3', 'fix_4', 'fix_5',
               'ttff_1', 'ttff_2', 'ttff_3', 'ttff_4', 'ttff_5',
               'anticipation', 'total_fix', 'mean_ttff', 'mean_ttff_excl1']
first_fix_cols = ['Participant', 'condition', 'Order', 'fix_1', 'ttff_1']

with pd.ExcelWriter('exports/biathlon_export.xlsx') as writer:
    df_flat[export_cols].to_excel(writer, sheet_name='all_fixations', index=False)
    df_flat[first_fix_cols].to_excel(writer, sheet_name='first_fixation', index=False)

fig_box.savefig('exports/boxplots.png', dpi=150, bbox_inches='tight')
fig_hist.savefig('exports/anticipation_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print('Exported:')
print(f'  biathlon_export.xlsx — all_fixations ({len(export_cols)} cols), first_fixation ({len(first_fix_cols)} cols)')
print('  boxplots.png')
print('  anticipation_histogram.png')
